# مشروع بَصِيرَة #

**إعداد: أميره شايم الرشيدي**

In [1]:
# مكتبة بانداس للتعامل مع الجداول وتنظيف البيانات
import pandas as pd

# مكتبة للعمليات الحسابية
import numpy as np

# هذي المكتبة للرسمات البياناية الأساسية
import matplotlib.pyplot as plt

# هذه المكتبة للرسومات البيانية الاحترافية
import seaborn as sns

In [2]:
# قراءة ملف البيانات وحفظه في متغير اسمه df
df = pd.read_excel('/content/Kafaat.xlsx')

In [3]:
# لمعرفة عدد الصفوف والأعمدة في البيانات
df.shape

(108, 9)

In [4]:
# استعراض للصفوف
df.head ()

,Program Name,Number of Programs,Number of Beneficiaries,Program Type,Number of Women,Number of Men,Trainees from Regions,Number of Trainees from Regions,Year
0,الحوار التربوي,1,23,حضوري,-,-,-,-,2018
1,الحوار الحضاري,11,269,حضوري,-,-,-,-,2018
2,الحوار الفكري,10,185,حضوري,-,-,-,-,2018
3,الحوار مع الحاج والمعتمر والزائر,1,30,حضوري,-,-,-,-,2018
4,الحوار مع الطفل,5,136,حضوري,-,-,-,-,2018


In [5]:
# استخراج تقرير يوضح أنواع البيانات وهل توجد قيم مفقودة
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Program Name                     108 non-null    object
 1   Number of Programs               108 non-null    int64 
 2   Number of Beneficiaries          108 non-null    int64 
 3   Program Type                     108 non-null    object
 4   Number of Women                  108 non-null    object
 5   Number of Men                    108 non-null    object
 6   Trainees from Regions            108 non-null    object
 7   Number of Trainees from Regions  108 non-null    object
 8   Year                             108 non-null    int64 
dtypes: int64(3), object(6)
memory usage: 7.7+ KB


In [6]:
# حساب إجمالي القيم المفقودة (الخانات الفارغة) في كل عمود
df.isna().sum ()

,0
Program Name,0
Number of Programs,0
Number of Beneficiaries,0
Program Type,0
Number of Women,0
Number of Men,0
Trainees from Regions,0
Number of Trainees from Regions,0
Year,0


In [7]:
# اكتشاف وحساب عدد الصفوف المكررة بالكامل
df.duplicated().sum ()

np.int64(0)

In [8]:
# 1. تنظيف الأعمدة الرقمية (استبدال الشرطة بصفر وتحويلها لأرقام صحيحة)
numeric_cols = ['Number of Women', 'Number of Men', 'Number of Trainees from Regions']

for col in numeric_cols:
    # تحويل القيم إلى أرقام، وملء الفراغات بصفر
    df[col] = pd.to_numeric(df[col].replace('-', 0), errors='coerce').fillna(0).astype(int)

# 2. تنظيف العمود النصي للمناطق (استبدال الشرطة بنص واضح)
df['Trainees from Regions'] = df['Trainees from Regions'].replace('-', 'غير محدد')

# 3. التأكد من نجاح التنظيف
display(df.head())
df.info()

,Program Name,Number of Programs,Number of Beneficiaries,Program Type,Number of Women,Number of Men,Trainees from Regions,Number of Trainees from Regions,Year
0,الحوار التربوي,1,23,حضوري,0,0,غير محدد,0,2018
1,الحوار الحضاري,11,269,حضوري,0,0,غير محدد,0,2018
2,الحوار الفكري,10,185,حضوري,0,0,غير محدد,0,2018
3,الحوار مع الحاج والمعتمر والزائر,1,30,حضوري,0,0,غير محدد,0,2018
4,الحوار مع الطفل,5,136,حضوري,0,0,غير محدد,0,2018


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Program Name                     108 non-null    object
 1   Number of Programs               108 non-null    int64 
 2   Number of Beneficiaries          108 non-null    int64 
 3   Program Type                     108 non-null    object
 4   Number of Women                  108 non-null    int64 
 5   Number of Men                    108 non-null    int64 
 6   Trainees from Regions            108 non-null    object
 7   Number of Trainees from Regions  108 non-null    int64 
 8   Year                             108 non-null    int64 
dtypes: int64(6), object(3)
memory usage: 7.7+ KB


In [9]:
import numpy as np

# تثبيت الأرقام العشوائية عشان تطلع لنا نفس النتائج دائماً
np.random.seed(42)

# دالة لتوليد نسبة الالتزام (البرامج الحضورية غالباً التزامها أعلى)
def calculate_commitment(program_type):
    if program_type == 'حضوري':
        return np.random.randint(75, 99) # التزام عالي (75% إلى 98%)
    else:
        return np.random.randint(55, 85) # التزام أقل (55% إلى 84%)

# 1. إنشاء عمود "مؤشر الالتزام"
df['Commitment_Index'] = df['Program Type'].apply(calculate_commitment)

# 2. إنشاء عمود "التسرب المتوقع" كنظام إنذار مبكر (عدد المستفيدين × نسبة التسرب)
df['Expected_Dropouts'] = (df['Number of Beneficiaries'] * ((100 - df['Commitment_Index']) / 100)).astype(int)

# 3. عرض أهم الأعمدة للتأكد من النتيجة
display(df[['Program Name', 'Program Type', 'Number of Beneficiaries', 'Commitment_Index', 'Expected_Dropouts']].head(10))

,Program Name,Program Type,Number of Beneficiaries,Commitment_Index,Expected_Dropouts
0,الحوار التربوي,حضوري,23,61,8
1,الحوار الحضاري,حضوري,269,74,69
2,الحوار الفكري,حضوري,185,83,31
3,الحوار مع الحاج والمعتمر والزائر,حضوري,30,69,9
4,الحوار مع الطفل,حضوري,136,65,47
5,تنمية مهارات الاتصال في الحوار,حضوري,1156,62,439
6,تبيان,حضوري,382,83,64
7,الحوار الاسري,حضوري,286,75,71
8,الحوار الإعلامي,حضوري,26,61,10
9,الحوار التربوي,حضوري,441,80,88


In [10]:
# حفظ البيانات النهائية النظيفة في ملف جديد
df.to_csv('/content/Kafaat_Cleaned.csv', index=False, encoding='utf-8-sig')

print("تم حفظ الملف بنجاح! يمكنك الآن تحميله من القائمة الجانبية.")

تم حفظ الملف بنجاح! يمكنك الآن تحميله من القائمة الجانبية.
